# Chapter 5 — Question 6
### Group 4

**Question:** We continue to consider the use of a logistic regression model to predict the probability of `default` using `income` and `balance` on the `Default` data set. We will now compute estimates for the standard errors of the `income` and `balance` logistic regression coefficients in two different ways: (1) using the bootstrap, and (2) using the standard formula for computing the standard errors in the `sm.GLM()` function.

## Data Importation

In [1]:
import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import ModelSpec as MS, summarize

Default = load_data('Default')
Default.head()

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


## (a) Standard errors using `summarize()` and `sm.GLM()`

We fit a multiple logistic regression model of `default` on `income` and `balance`, and read off the standard errors reported by `statsmodels`.

In [2]:
X = MS(['income', 'balance']).fit_transform(Default)
y = (Default['default'] == 'Yes').astype(float)   # 1 = default, 0 = no default

glm_model = sm.GLM(y, X, family=sm.families.Binomial())
glm_results = glm_model.fit()
summarize(glm_results)

,coef,std err,z,P>|z|
intercept,-11.540500,0.435000,-26.544,0.0
income,0.000021,0.000005,4.174,0.0
balance,0.005600,0.000000,24.835,0.0


The `std err` column above gives the formula-based standard errors for `income` and `balance`. These come from the estimated covariance matrix of the maximum likelihood estimator, which relies on standard asymptotic theory (i.e. large-sample approximations).

## (b) `boot_fn()`

A function that takes the `Default` data set and a vector of indices, refits the logistic regression model on `data.iloc[idx]`, and returns the `income` and `balance` coefficient estimates.

In [3]:
def boot_fn(data, idx):
    """Fit logistic regression of default ~ income + balance on data.iloc[idx],
    returning the income and balance coefficient estimates."""
    D = data.iloc[idx]
    X_ = MS(['income', 'balance']).fit_transform(D)
    y_ = (D['default'] == 'Yes').astype(float)
    results_ = sm.GLM(y_, X_, family=sm.families.Binomial()).fit()
    return results_.params[['income', 'balance']]

# Sanity check: running boot_fn on the full (unresampled) index set
# should reproduce the coefficients from part (a)
full_idx = np.arange(Default.shape[0])
boot_fn(Default, full_idx)

income     0.000021
balance    0.005647
dtype: float64

## (c) Estimating standard errors via the bootstrap

We repeatedly generate bootstrap samples (resampling `n` observations **with replacement**), apply `boot_fn()` to each, and compute the standard deviation of the resulting coefficient estimates across `B = 1000` replications.

In [4]:
rng = np.random.default_rng(0)   # set seed for reproducibility
n = Default.shape[0]
B = 1000

boot_coefs = np.zeros((B, 2))
for b in range(B):
    idx = rng.choice(n, size=n, replace=True)
    coefs = boot_fn(Default, idx)
    boot_coefs[b, 0] = coefs['income']
    boot_coefs[b, 1] = coefs['balance']

boot_se_income  = boot_coefs[:, 0].std(ddof=1)
boot_se_balance = boot_coefs[:, 1].std(ddof=1)

print(f"Bootstrap SE (income) : {boot_se_income:.8f}")
print(f"Bootstrap SE (balance): {boot_se_balance:.8f}")

Bootstrap SE (income) : 0.00000477
Bootstrap SE (balance): 0.00023055


## (d) Comparing the two sets of standard errors

In [5]:
import pandas as pd

comparison = pd.DataFrame({
    'GLM formula SE': [glm_results.bse['income'], glm_results.bse['balance']],
    'Bootstrap SE':   [boot_se_income, boot_se_balance]
}, index=['income', 'balance'])

comparison

,GLM formula SE,Bootstrap SE
income,0.000005,0.000005
balance,0.000227,0.000231


**Comment:**

The bootstrap standard errors and the `sm.GLM()` formula-based standard errors are very close to one another (within roughly 1-5%) for both `income` and `balance`.

This agreement is expected. The formula used by `sm.GLM()` relies on asymptotic theory for maximum likelihood estimation — it assumes the model is correctly specified and that the sample size is large enough for the sampling distribution of the coefficients to be approximately normal. With `n = 10,000` observations, these assumptions hold reasonably well, so the two approaches converge to similar answers.

The key conceptual difference is that the bootstrap standard errors are obtained **without relying on these distributional assumptions** — they come purely from resampling the observed data and directly measuring how much the coefficient estimates vary across resamples. This makes the bootstrap especially valuable in settings where the standard formula's assumptions are questionable (e.g., small sample sizes, or models where no formula for the standard error exists at all). The fact that the two methods agree closely here serves as a useful validation of both approaches for this particular problem.